In [1]:
#******************************************************
#*
#* Name:         nb-06-python-polars-csv-files
#*     
#* Design Phase:
#*     Author:   John Miner
#*     Date:     04-01-2026
#*     Purpose:  Mount storage and change compute.
#*               Use polars to read/write to delta.
#* 
#******************************************************/

In [2]:
%%configure -f
{
    "vCores": 2,   
    "defaultLakehouse": {  
        "name": "lh_python_vs_spark",
        "id": "f880ba3e-79f9-4297-8b7f-4e39ca3d158b",
        "workspaceId": "9d12d1b1-e32e-4bbe-9753-f0715d21ad49" 
    },
    "mountPoints": [
        {
            "mountPoint": "/mystockdata",
            "source": "abfss://raw@sa4adls2030.dfs.core.windows.net/stocks"
        }
    ],
}

In [3]:
%%capture

#
# 0 - get lakehouse abfs path
#

def get_lakehouse_abfs_path(key = "/default"):
    
    # get mount point
    try:
        path = list(filter(lambda x: x["mountPoint"] == key, notebookutils.fs.mounts()))[0]["source"]
    except:
        path = ""

    # return results
    return path

# get the path
lakehouse_path = get_lakehouse_abfs_path()

In [4]:
#
#  debug - show path
#

print(lakehouse_path)


abfss://9d12d1b1-e32e-4bbe-9753-f0715d21ad49@onelake.dfs.fabric.microsoft.com/f880ba3e-79f9-4297-8b7f-4e39ca3d158b


In [5]:
#
#  1 - create function to read files recursively
#

import polars as pl

def read_csv_files(path: str, custom_schema: dict) -> pl.DataFrame:

    # read into one frame
    df = pl.scan_csv(path, schema=custom_schema, has_header=True).collect()

    # found files
    if ~ df.is_empty():
        return df

    # no files found 
    else:
        return pl.DataFrame() 


In [6]:
#
#  2 - execute the function (execution time = 48 s)
#

import os
import polars as pl

# get path
path1 = notebookutils.fs.getMountPath('/mystockdata')

# list folders
contents1 = os.listdir(path1)

# make schema
schema1 = {
    "st_symbol": pl.String,
    "st_date": pl.String,
    "st_open": pl.Float64,
    "st_high": pl.Float64,
    "st_low": pl.Float64,
    "st_close": pl.Float64,
    "st_adjclose": pl.Float64,
    "st_volume": pl.Int64,
}

# make array by year
frames = []

# for each folder
for item1 in contents1:
    if (item1 != 'ALL'):

        # show progress
        print(f"processing folder {item1}")

        # full path
        path2 = path1 + '/' + item1 + '/*.CSV'

        # read files
        df1 = read_csv_files(path2, schema1)

        # append to array
        frames.append(df1)

df1 = pl.concat(frames, how="vertical")     


processing folder S&P-2013
processing folder S&P-2014
processing folder S&P-2015
processing folder S&P-2016
processing folder S&P-2017


In [7]:
#
#  3 - Show rows of data
#

display(df1.head(10))

In [8]:
#
# 4 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "polars_stock_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [9]:
#
# 5 - write delta lake table (5 sec)
#

from deltalake import write_deltalake

# remove dividends
df2 = df1.filter(pl.col("st_volume") != 0)

# over write 
write_deltalake(table_path, df1.to_pandas(), mode='overwrite')


In [10]:
#
# 6 - read a single file (3 sec)
#

import polars as pl

path ='/lakehouse/default/Files/SNP-5YEARS.CSV'
df3 = pl.read_csv(path, separator='\t', has_header=True)
print(df3)


shape: (626_714, 8)
┌───────────┬────────────┬──────────┬──────────┬──────────┬──────────┬─────────────┬───────────┐
│ st_symbol ┆ st_date    ┆ st_open  ┆ st_high  ┆ st_low   ┆ st_close ┆ st_adjclose ┆ st_volume │
│ ---       ┆ ---        ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---         ┆ ---       │
│ str       ┆ str        ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64         ┆ i64       │
╞═══════════╪════════════╪══════════╪══════════╪══════════╪══════════╪═════════════╪═══════════╡
│ A         ┆ 12/31/2013 ┆ 41.09442 ┆ 41.16595 ┆ 40.82976 ┆ 40.90844 ┆ 39.39425    ┆ 1316000   │
│ A         ┆ 12/30/2013 ┆ 40.8083  ┆ 41.10157 ┆ 40.71531 ┆ 41.00143 ┆ 39.4838     ┆ 1576900   │
│ A         ┆ 12/27/2013 ┆ 40.93705 ┆ 41.07296 ┆ 40.85837 ┆ 40.89413 ┆ 39.38047    ┆ 913000    │
│ A         ┆ 12/26/2013 ┆ 40.94421 ┆ 41.2804  ┆ 40.94421 ┆ 41.10157 ┆ 39.48931    ┆ 998700    │
│ A         ┆ 12/24/2013 ┆ 41.13018 ┆ 41.20887 ┆ 40.89413 ┆ 40.93705 ┆ 39.33123    ┆ 1090000   │
│ …       

In [11]:
#
# 7 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "polars_one_stock_file"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [12]:
#
# 8 - write delta lake table (3 sec)
#

# must be adls path
df3.write_delta(table_path, mode='overwrite')


In [1]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse

-- may take some time to refresh
-- select * from sys.objects where is_ms_shipped = 0

select top 5 * from bronze.polars_stock_data;

In [4]:
#
# 9 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "polars_weather_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [5]:
#
# 10 - read + transform weather data
#

import polars as pl
import polars.selectors as cs

# read csv files
df4 = pl.scan_csv(f"{lakehouse_path}/Files/weather/*csv", include_file_paths="path").collect()

# break into two frames, join on date
df5 = df4.filter(pl.col("path").str.contains("high_temps.csv"))
df6 = df4.filter(pl.col("path").str.contains("low_temps.csv"))
ret = df6.join(df5, on="date", how="inner")

# rename & drop columsn
ret = ret.drop(cs.by_index(4)) 
ret = ret.drop(cs.by_index(2)) 
ret = ret.rename({ret.columns[0]: "obs_date"})
ret = ret.rename({ret.columns[1]: "obs_low_temp"})
ret = ret.rename({ret.columns[2]: "obs_high_temp"})

# show the final df
display(ret.head(5))


In [6]:
#
# 8 - write delta lake table (3 sec)
#

# must be adls path
ret.write_delta(table_path, mode='overwrite')